# 商品明细与描述处理

按大小写敏感的商品编码分别统计商品事实和描述。全量有效明细提供金额、数量和去重发票数。非空描述子集仅用于生成展示映射，按编码左连接。

去除首尾空白并转小写只用于比较描述。沿用既有冲突判定，最高频候选的占比按原规则保留三位小数，达到 70% 且该编码至少有 5 条可用描述时，采用暂定主描述。候选频次并列时按规范化文本升序，原始写法按各自频次降序、原文升序选取。结果不依赖明细行顺序。

展示名称取选中的原始文本，保留其大小写和空白。待复核编码的明细保留原始描述。完全没有可用描述的编码也保留事实指标，并标记为缺描述待复核。

In [1]:
import pandas as pd

data_10 = pd.read_csv(
    '../data/processed/data_10month.csv',
    dtype={'stockcode': 'string', 'description': 'string'},
    keep_default_na=False,
    na_values=[''],
)

In [2]:
data_10c = data_10.copy()

In [3]:
# 列名规范化
data_10c.columns = (
    data_10c.columns
    .str.strip()
    .str.lower()
)
data_10c.columns

Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country', 'lineamount'],
      dtype='object')

In [4]:
# 数据转换
data_10c['invoicedate'] = pd.to_datetime(data_10c['invoicedate'])

In [5]:
# 检查数据
a,b = data_10c.shape
print(f'10月份数据为{a}行，{b}列') # 57834 9
time_min = data_10c['invoicedate'].min()
time_max = data_10c['invoicedate'].max()
print(f'最小日期为{time_min}，最大日期为{time_max}') # 2011-10-02 10:32:00 2011-10-31 17:13:00

10月份数据为57834行，9列
最小日期为2011-10-02 10:32:00，最大日期为2011-10-31 17:13:00


In [6]:
# 检查 stockcode 字段的缺失情况
stockcode_n = data_10c['stockcode'].isna().sum()
print(f'stockcode字段缺失情况为{stockcode_n}行') # 0
# 检查 description 字段的缺失情况
description_n = data_10c['description'].isna().sum()
print(f'description字段缺失情况为{description_n}行') # 0

stockcode字段缺失情况为0行
description字段缺失情况为0行


In [7]:
# 先排除 description 为空的情况
data_10c_dn = data_10c[data_10c['description'].isna()]
data_10c_d = data_10c[~(data_10c['description'].isna())]
print(f"为空{len(data_10c_dn)}行，非空{len(data_10c_d)}行") # 0 , 57834 共计57834行

为空0行，非空57834行


In [8]:
# 对 description 非空情况进行分析，检查同一商品 description 不同的情况
data_10c_dc = data_10c_d.copy()
# 去除两端空白并转为小写，仅用于第一次冲突检查。
data_10c_dc['description_clean'] = (
    data_10c_dc['description'].astype('string')
    .str.strip()
    .str.lower()
)
# 排除缺失或空字符串描述。
data_10c_dc = data_10c_dc[
    data_10c_dc['description_clean'].notna()
    &
    data_10c_dc['description_clean'].ne('')].copy()
print(f"去除空值与空字符串后剩余{len(data_10c_dc)}行数据") # 57834

去除空值与空字符串后剩余57834行数据


In [9]:
def check_description_conflicts(data: pd.DataFrame, description_column: str, count_column: str) -> pd.DataFrame:
    """按精确编码统计 data 中的指定描述列，返回冲突候选及其指定列名的频次。"""
    counts = data.groupby(['stockcode', description_column]).size().reset_index(name=count_column)
    summary = (
        counts.groupby('stockcode').agg(
            description_variants=(description_column, 'nunique'),
            totalcount=(count_column, 'sum'),
        ).reset_index()
    )
    conflict_codes = summary[summary['description_variants'] > 1]
    return counts[counts['stockcode'].isin(conflict_codes['stockcode'])].copy()


description_conflicts = check_description_conflicts(data_10c_dc, 'description_clean', 'description_count')
print(f"共有{description_conflicts['stockcode'].nunique()}件商品描述不一致")

description_conflicts.head(5)

共有62件商品描述不一致


,stockcode,description_clean,description_count
313,21232,strawberry ceramic trinket box,57
314,21232,strawberry ceramic trinket pot,13
1016,22416,set of 36 doilies spaceboy design,6
1017,22416,set of 36 spaceboy paper doilies,2
1058,22466,fairy tale cottage night light,29


In [10]:

# 计算存在描述矛盾的商品的描述在该商品中的占比
# 计算每种商品的数量
data_10c_count = data_10c_dc.groupby('stockcode').size().reset_index(name='number')
description_conflicts_mer = pd.merge(description_conflicts,data_10c_count,
                                     on='stockcode',
                                     how='left').copy()
description_conflicts_mer['percentage'] = round((description_conflicts_mer['description_count'] / description_conflicts_mer['number'])* 100,3)
description_conflicts_mer.head(5)

,stockcode,description_clean,description_count,number,percentage
0,21232,strawberry ceramic trinket box,57,70,81.429
1,21232,strawberry ceramic trinket pot,13,70,18.571
2,22416,set of 36 doilies spaceboy design,6,8,75.000
3,22416,set of 36 spaceboy paper doilies,2,8,25.000
4,22466,fairy tale cottage night light,29,30,96.667


In [11]:
# 对冲突候选先按频次排序，再按规范化文本确定并列顺序。
description_ranked = description_conflicts_mer.sort_values(
    ['stockcode', 'description_count', 'description_clean'],
    ascending=[True, False, True],
)
description_choice = description_ranked.drop_duplicates('stockcode').copy()
description_choice['is_clear'] = (
    description_choice['percentage'].ge(70)
    & description_choice['number'].ge(5)
)

# 在同一规范化候选的原始写法中，取最常见的原文。并列时按原文升序。
original_choices = (
    data_10c_dc.groupby(['stockcode', 'description_clean', 'description'])
    .size()
    .reset_index(name='original_count')
    .sort_values(
        ['stockcode', 'description_clean', 'original_count', 'description'],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(['stockcode', 'description_clean'])
    .rename(columns={'description': 'description_final'})
)
description_choice = description_choice.merge(
    original_choices[['stockcode', 'description_clean', 'description_final']],
    on=['stockcode', 'description_clean'],
    validate='one_to_one',
)
display(description_choice.head())

,stockcode,description_clean,description_count,number,percentage,is_clear,description_final
0,21232,strawberry ceramic trinket box,57,70,81.429,True,STRAWBERRY CERAMIC TRINKET BOX
1,22416,set of 36 doilies spaceboy design,6,8,75.000,True,SET OF 36 DOILIES SPACEBOY DESIGN
2,22466,fairy tale cottage night light,29,30,96.667,True,FAIRY TALE COTTAGE NIGHT LIGHT
3,22776,sweetheart 3 tier cake stand,47,48,97.917,True,SWEETHEART 3 TIER CAKE STAND
4,22804,pink hanging heart t-light holder,21,33,63.636,False,PINK HANGING HEART T-LIGHT HOLDER


In [12]:
# 描述映射每个精确编码最多一行。未决冲突不加入主描述的映射。
mapping_columns = ['stockcode', 'description_final']
unchanged = original_choices.loc[
    ~original_choices['stockcode'].isin(description_choice['stockcode']),
    mapping_columns,
].assign(description_status='沿用原描述')
tentative = description_choice.loc[
    description_choice['is_clear'], mapping_columns
].assign(description_status='暂定主描述')
display_choices = pd.concat([unchanged, tentative], ignore_index=True)
pending_conflicts = description_choice.loc[
    ~description_choice['is_clear'], 'stockcode'
]

In [13]:
def summarize_product_facts(detail: pd.DataFrame) -> pd.DataFrame:
    # 每张发票在同一精确编码内只计一次。
    return detail.groupby('stockcode', dropna=False).agg(
        detail_row_count=('stockcode', 'size'),
        gross_amount_gbp=('lineamount', 'sum'),
        units=('quantity', 'sum'),
        invoice_coverage=('invoiceno', 'nunique'),
    ).reset_index()

product_facts = summarize_product_facts(data_10c)
product_summary = product_facts.merge(
    display_choices, on='stockcode', how='left', validate='one_to_one'
)
product_summary.loc[
    product_summary['stockcode'].isin(pending_conflicts), 'description_status'
] = '待复核'
product_summary['description_status'] = (
    product_summary['description_status'].fillna('缺描述待复核')
)
review_label = '待复核 · ' + product_summary['stockcode']
product_summary['display_name'] = product_summary['description_final'].fillna(review_label)

In [14]:
# 入库使用原有列。原始描述保留在上游有效明细和这里的 description_raw。
detail_with_display = data_10c.rename(columns={'description': 'description_raw'}).merge(
    display_choices[mapping_columns], on='stockcode', how='left', validate='many_to_one'
)
detail_with_display['description'] = detail_with_display['description_final'].fillna(
    detail_with_display['description_raw']
)
result = detail_with_display[
    ['invoiceno', 'stockcode', 'quantity', 'invoicedate', 'unitprice',
     'customerid', 'country', 'description', 'lineamount']
]

In [15]:
# 比较连接后的事实，并核对供 MySQL 导入的明细重新聚合后的事实。
mapped_facts = product_summary[product_facts.columns]
exported_facts = summarize_product_facts(result)
pd.testing.assert_frame_equal(product_facts, mapped_facts)
pd.testing.assert_frame_equal(product_facts, exported_facts)

fact_check = pd.DataFrame({
    '合并前': product_facts.drop(columns='stockcode').sum(),
    '合并后': mapped_facts.drop(columns='stockcode').sum(),
}).round(2)
fact_check.loc['精确商品编码数'] = [len(product_facts), len(mapped_facts)]
display(fact_check)
display(product_summary['description_status'].value_counts().rename('编码数'))

,合并前,合并后
detail_row_count,57834.0,57834.0
gross_amount_gbp,1148101.1,1148101.1
units,619049.0,619049.0
invoice_coverage,57065.0,57065.0
精确商品编码数,2854.0,2854.0


description_status
沿用原描述    2792
暂定主描述      44
待复核        18
Name: 编码数, dtype: int64

In [16]:
pending_codes = product_summary.loc[
    product_summary['description_status'].isin(['待复核', '缺描述待复核']),
    'stockcode',
]
result_pending = result[result['stockcode'].isin(pending_codes)]
print(f"待复核商品编码：{len(pending_codes):,} 个，明细：{len(result_pending):,} 行")

待复核商品编码：18 个，明细：586 行


In [17]:
# 导出结果
result.to_csv('../data/processed/data_10_product.csv',index=False)
result_pending.to_csv('../data/processed/data_10_product_pending.csv',index=False)

## 结果与复核范围

输入为[10 月有效明细（本地生成）](../data_source.md#本地生成文件)。原始描述可从该文件回溯，商品入库 CSV 的 description 字段供 SQL 和 Power BI 展示。待复核 CSV 保留未决或缺描述商品的明细。商品发票覆盖求和是各编码覆盖数相加。

[商品排名与对账](../reports/商品排名与对账/商品排名对账.md)包含历史证据快照。重新运行会生成本地商品明细与待复核 CSV，历史展示映射 CSV 保留当时的显示文本。